# Hierarchical clustering using Wasserstein distance

It does not rely on either clustering or dimensionality reduction. We are operating directly on the latent embeddings.

In [1]:
import sys
sys.path.append('../')

import numpy as np
import ot
import matplotlib as mpl
mpl.rcParams['legend.markerscale'] = 1.5
import matplotlib.pyplot as plt
import warnings
from tqdm import tqdm
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from utils.mypath import MyPath
from utils.mypalette import mutation_palette, condition_palette, day_palette
from sklearnex import patch_sklearn
from utils.start_notebook import setup_experiment
patch_sklearn()
  
warnings.filterwarnings('ignore')

single_exps = ['E1', 'E4', 'E2']
x = '240813_5'

annotations_df, raw_emb = setup_experiment(x, single_exps)
annotations_df['mut_cond'] = annotations_df['mut_cond'].str.replace('_', '\n')
als_lab = ['CTRL', 'ALS']

ModuleNotFoundError: No module named 'utils.config'

## Computations separated by batch

In [ ]:
for b in single_exps:
    filtered_annotations_df = annotations_df[annotations_df['experiment'] == b]
    filtered_embeddings = raw_emb[filtered_annotations_df.index]
    print(filtered_annotations_df.shape)
    print(filtered_embeddings.shape)
    # Compute pairwise Wasserstein distances between mut_cond labels
    mut_cond_labels = filtered_annotations_df['mut_cond'].unique()
    pairwise_distances = np.zeros((len(mut_cond_labels), len(mut_cond_labels)))

    for i, label_1 in enumerate(mut_cond_labels):
        for j, label_2 in enumerate(mut_cond_labels):
            if i < j:  # Compute only for upper triangle
                pts1 = filtered_embeddings[filtered_annotations_df['mut_cond'] == label_1]
                pts2 = filtered_embeddings[filtered_annotations_df['mut_cond'] == label_2]
                pairwise_distances[i, j] = ot.emd2(ot.unif(len(pts1)), ot.unif(len(pts2)), ot.dist(pts1, pts2))
                pairwise_distances[j, i] = pairwise_distances[i, j]  # Symmetric matrix

    # Perform hierarchical clustering
    linkage_matrix = linkage(squareform(pairwise_distances), method='ward')

    # Plot dendrogram
    plt.figure(figsize=(8, 8))
    dendrogram(
        linkage_matrix,
        labels=mut_cond_labels,
        leaf_rotation=0,
        leaf_font_size=10,
        orientation='right',
        link_color_func=lambda k: 'black'
    )

    # Color the labels based on the part before '_'
    ax = plt.gca()
    for label in ax.get_yticklabels():
        text = label.get_text()
        prefix = text.split('\n')[0]
        label.set_color(mutation_palette.get(prefix, 'black'))
    plt.title(f'Wasserstein distance for mutation, treatment classes - {b}')
    plt.ylabel('Mutation and treatment')
    plt.xlabel('Wasserstein Distance')
    plt.tight_layout()
    #plt.savefig(os.path.join(saving_dir, f'dendrogram_{b}.pdf'), bbox_inches='tight', dpi=300)
    plt.show()

annotations_df['mut_cond_exp'] = annotations_df['mut_cond'] + '\n' + annotations_df['experiment']
global_labels = annotations_df['mut_cond_exp'].unique()
pairwise_distances = np.zeros((len(global_labels), len(global_labels)))
for i, label_1 in enumerate(global_labels):
    for j, label_2 in enumerate(global_labels):
        if i < j:  # Compute only for upper triangle
            pts1 = raw_emb[annotations_df['mut_cond_exp'] == label_1]
            pts2 = raw_emb[annotations_df['mut_cond_exp'] == label_2]
            pairwise_distances[i, j] = ot.emd2(ot.unif(len(pts1)), ot.unif(len(pts2)), ot.dist(pts1, pts2))
            pairwise_distances[j, i] = pairwise_distances[i, j]  # Symmetric matrix
# Perform hierarchical clustering
linkage_matrix = linkage(squareform(pairwise_distances), method='ward')
# Plot dendrogram
fig, axes = plt.subplots(1, 3, figsize=(20, 12), sharey=True)

# Dendrogram colored by mutation
dendrogram(
    linkage_matrix,
    labels=global_labels,
    leaf_rotation=0,
    leaf_font_size=10,
    orientation='right',
    link_color_func=lambda k: 'black',
    ax=axes[0]
)
for label in axes[0].get_yticklabels():
    text = label.get_text()
    prefix = text.split('\n')[0]
    label.set_color(mutation_palette.get(prefix, 'black'))
axes[0].set_title('Colored by Mutation')
axes[0].set_xlabel('Wasserstein Distance')
axes[0].set_ylabel('Mutation, Treatment, Experiment')

# Dendrogram colored by condition
dendrogram(
    linkage_matrix,
    labels=global_labels,
    leaf_rotation=0,
    leaf_font_size=10,
    orientation='right',
    link_color_func=lambda k: 'black',
    ax=axes[1]
)
for label in axes[1].get_yticklabels():
    text = label.get_text()
    condition = text.split('\n')[1]
    label.set_color(condition_palette.get(condition, 'black'))
axes[1].set_title('Colored by Condition')
axes[1].set_xlabel('Wasserstein Distance')

# Dendrogram colored by experiment
dendrogram(
    linkage_matrix,
    labels=global_labels,
    leaf_rotation=0,
    leaf_font_size=10,
    orientation='right',
    link_color_func=lambda k: 'black',
    ax=axes[2]
)
for label in axes[2].get_yticklabels():
    text = label.get_text()
    experiment = text.split('\n')[-1]
    label.set_color(day_palette.get(experiment, 'black'))
axes[2].set_title('Colored by Experiment')
axes[2].set_xlabel('Wasserstein Distance')

plt.tight_layout()
#plt.savefig(os.path.join(saving_dir, 'dendrogram_global.pdf'), bbox_inches='tight', dpi=300)
plt.show()